In [1]:
import numpy as np
import pandas as pd
import random

In [ ]:
class GridCarEnv():
    def __init__(self, dims=(6,6), max_steps=200):
        self.dims = dims # (row, col)
        self.rewards = {
            'r_goal': 100, 
            'step': -1, 
            'hit_pitfall': -5,
            'hit_obstacle': -10,
            'bump_wall': -20}
        self.max_steps = max_steps
        self.start = (dims[0]-1, 0) # bottom left
        self.end = (0, dims[1]-1) # upper right
        self.positions = {'pitfalls': [(1,3),(4,1),(3,5)], 'obstacles': [(1,2),(1,4),(3,2), (4,3)]}
        self.pos = self.start
        self.reward_score = 0
        self.current_step = 0
        self.actions = { # 4 actions: 0=Up, 1=Down, 2=Left, 3=Right
            0: (-1,  0),
            1: ( 1,  0),
            2: ( 0, -1),
            3: ( 0,  1),
        }
    
    def reset(self):
        self.pos = self.start
        self.current_step = 0
        self.reward_score = 0
        return self.pos

    def step(self, move):

        reward = self.rewards['step']
        done = False
        self.current_step += 1
        if self.current_step == self.max_steps:
            done = True
            return self.pos, reward, done

        if move == 0:
            new_pos = (self.pos[0] + self.actions[0][0], self.pos[1]+ self.actions[0][1])
            if new_pos[0] == -1:
                reward += self.rewards['bump_wall']
            elif new_pos in self.positions['obstacles']:
                reward += self.rewards['hit_obstacle']
            else:
                self.pos = new_pos #change position only if it is not out of bounds
        elif move == 1:
            new_pos = (self.pos[0] + self.actions[1][0], self.pos[1]+ self.actions[1][1])
            if new_pos[0] == self.dims[0]:
                reward += self.rewards['bump_wall']
            elif new_pos in self.positions['obstacles']:
                reward += self.rewards['hit_obstacle']
            else:
                self.pos = new_pos
        elif move == 2:
            new_pos = (self.pos[0] + self.actions[2][0], self.pos[1]+ self.actions[2][1])
            if new_pos[1] == -1:
                reward += self.rewards['bump_wall']
            elif new_pos in self.positions['obstacles']:
                reward += self.rewards['hit_obstacle']
            else:
                self.pos = new_pos
        elif move == 3:
            new_pos = (self.pos[0] + self.actions[3][0], self.pos[1]+ self.actions[3][1])
            if new_pos[1] == self.dims[1]:
                reward += self.rewards['bump_wall']
            elif new_pos in self.positions['obstacles']:
                reward += self.rewards['hit_obstacle']
            else:
                self.pos = new_pos
        

        if self.pos in self.positions['pitfalls']:
            self.reward_score = self.reward_score + self.rewards['hit_pitfall']
            reward += self.rewards['hit_pitfall'] 
        elif self.pos == self.end:
            reward += self.rewards['r_goal']
            done = True

        return self.pos, reward, done
    

def initialize_q_table(n_states, n_actions):
    return np.zeros((n_states, n_actions))


def state_to_index(state, dims):
    return state[0] * dims[1] + state[1]


def update_q_table(Q, alpha, r, gamma, a, s_old, s, dims):
    s_old_index = state_to_index(s_old, dims)
    s_index = state_to_index(s, dims)

    Q[s_old_index, a] = Q[s_old_index, a] + alpha * (
        r + gamma * np.max(Q[s_index, :]) - Q[s_old_index, a]
    )

    return Q


def epsilon_greedy(Q, state, epsilon, dims):
    state_index = state_to_index(state, dims)

    if np.random.random() < epsilon:
        return np.random.choice(Q.shape[1])

    return np.argmax(Q[state_index, :])


def softmax(Q, state, temperature, dims):
    state_index = state_to_index(state, dims)
    q_values = Q[state_index, :]

    temperature = max(temperature, 1e-8)

    shifted_q_values = q_values - np.max(q_values)
    exp_values = np.exp(shifted_q_values / temperature)
    probabilities = exp_values / np.sum(exp_values)

    return np.random.choice(len(q_values), p=probabilities)


In [ ]:
def train(alpha, gamma, decay, episodes, policy="epsilon_greedy"):
    env = GridCarEnv()

    n_states = env.dims[0] * env.dims[1]
    n_actions = len(env.actions)

    Q = initialize_q_table(n_states, n_actions)

    epsilon = 1.0
    epsilon_min = 0.01

    temperature = 1.0
    temperature_min = 0.1

    reward_list = []
    step_list = []
    success_count = 0

    for episode in range(episodes):
        state = env.reset()
        total_reward = 0

        for step_count in range(env.max_steps):
            s_old = state

            if policy == "epsilon_greedy":
                action = epsilon_greedy(Q, s_old, epsilon, env.dims)

            elif policy == "softmax":
                action = softmax(Q, s_old, temperature, env.dims)

            else:
                raise ValueError("policy must be 'epsilon_greedy' or 'softmax'")

            pos, reward, done = env.step(action)

            Q = update_q_table(Q, alpha, reward, gamma, action, s_old, pos, env.dims)

            state = pos
            total_reward += reward

            if done:
                break

        if state == env.end:
            success_count += 1

        if policy == "epsilon_greedy":
            epsilon = max(epsilon_min, epsilon * decay)

        elif policy == "softmax":
            temperature = max(temperature_min, temperature * decay)

        reward_list.append(total_reward)
        step_list.append(step_count + 1)

    success_rate = (success_count / episodes) * 100

    return Q, reward_list, step_list, success_rate

In [ ]:
def train_loop(alphas, gammas, decay_values, episodes, policy="softmax"):
    results = {}

    for alpha in alphas:
        for gamma in gammas:
            for decay in decay_values:
                Q, reward_list, step_list, success_rate = train(
                    alpha=alpha,
                    gamma=gamma,
                    decay=decay,
                    episodes=episodes,
                    policy=policy
                )

                results[(policy, alpha, gamma, decay)] = {
                    "Q": Q,
                    "reward_list": reward_list,
                    "step_list": step_list,
                    "success_rate": success_rate
                }

                if policy == "epsilon_greedy":
                    decay_name = "epsilon_decay"
                else:
                    decay_name = "temperature_decay"

                print(
                    f"policy={policy}, alpha={alpha}, gamma={gamma}, "
                    f"{decay_name}={decay} | success rate={success_rate:.2f}% | "
                    f"avg reward={np.mean(reward_list):.2f}"
                )

    return results

In [ ]:
#import matplotlib.pyplot as plt

#plt.plot(reward_list)
#plt.xlabel('Episode')
#plt.ylabel('Total Reward')
#plt.title('Q-Learning Training Performance')
#plt.show()

NameError: name 'reward_list' is not defined

In [10]:
policy = "softmax"

alphas = [0.1, 0.3, 0.9]
gammas = [0.5, 0.9, 0.99]

decay_values = [0.99, 0.995, 0.999]

episodes = 5000

temperature = 1.0
temperature_min = 0.1

def train_loop(alpha, gamma, epsilon_decay=0.995, temperature_decay=0.995, episodes=5000, policy="epsilon_greedy"):
    env = GridCarEnv()

    n_states = env.dims[0] * env.dims[1]
    n_actions = len(env.actions)

    Q = initialize_q_table(n_states, n_actions)

    epsilon = 1.0
    epsilon_min = 0.01

    temperature = 1.0
    temperature_min = 0.1

    reward_list = []
    success_count = 0

    for episode in range(episodes):
        state = env.reset()
        total_reward = 0

        for step in range(env.max_steps):
            s_old = state

            if policy == "epsilon_greedy":
                if np.random.random() < epsilon:
                    action = np.random.choice(n_actions)
                else:
                    s_old_index = state_to_index(s_old, env.dims)
                    action = np.argmax(Q[s_old_index, :])

            elif policy == "softmax":
                action = softmax(Q, s_old, temperature, env.dims)

            pos, reward, done = env.step(action)

            Q = update_q_table(Q, alpha, reward, gamma, action, s_old, pos, env.dims)

            state = pos
            total_reward += reward

            if done:
                break

        if state == env.end:
            success_count += 1

        if policy == "epsilon_greedy":
            epsilon = max(epsilon_min, epsilon * epsilon_decay)

        elif policy == "softmax":
            temperature = max(temperature_min, temperature * temperature_decay)

        reward_list.append(total_reward)

    success_rate = (success_count / episodes) * 100

    return reward_list, success_rate

In [11]:
results = train_loop(alphas, gammas, decay_values, episodes, policy="softmax")

TypeError: can't multiply sequence by non-int of type 'numpy.float64'

In [ ]:
results_eps = train_loop(alphas, gammas, decay_values, episodes, policy="epsilon_greedy")

In [7]:
env = GridCarEnv()

n_states = env.dims[0] * env.dims[1]  # 6*6 = 36
n_actions = len(env.actions)          # 4

policy = "softmax"

decay_values = [0.99, 0.995, 0.999]

results = train_loop(
    alphas=alphas,
    gammas=gammas,
    decay_values=decay_values,
    episodes=episodes,
    policy=policy
)

TypeError: train() got an unexpected keyword argument 'temperature_decay'